# **Import Libraries**
___

In [1]:
import numpy as np
import pandas as pd
import os

import seaborn as sns
import matplotlib.pyplot as plt

# **Load Data**
_____

In [3]:
os.chdir('/Users/dannyhogan/Desktop/ST-498')
UsData= pd.read_csv("Data/MacroVariablesUS.csv",index_col=0)
UkData= pd.read_csv("Data/MacroVariablesUK.csv",index_col=0)

In [5]:
UsData = UsData.drop(columns=['US_Credit_lags'])

In [15]:
def lag_all_except(df, exclude_col, n_lags=1, drop_na=True, drop_original=False):
    """
    Creates lagged versions of every column except `exclude_col`.

    Parameters
    ----------
    df             : your DataFrame, sorted by date already
    exclude_col    : column to leave untouched (usually your target)
    n_lags         : how many lags to create per column (default 1)
                     e.g. n_lags=3 creates _lag1, _lag2, _lag3 for each column
    drop_na        : whether to drop rows with NaN introduced by lagging (default True)
    drop_original  : whether to drop the original unlagged columns after creating
                     the lagged versions (default False)
                     e.g. income becomes income_lag1, income_lag2 — income itself is removed

    Returns
    -------
    A new DataFrame with the original columns plus the lagged ones.
    Your exclude_col is carried through unchanged.
    """

    df = df.copy()
    cols_to_lag = [c for c in df.columns if c != exclude_col]

    for col in cols_to_lag:
        for lag in range(1, n_lags + 1):
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    if drop_original:
        df = df.drop(columns=cols_to_lag)

    if drop_na:
        df = df.dropna().reset_index(drop=True)

    return df

In [16]:
df = lag_all_except(UsData, exclude_col="US_Credit", n_lags=1,drop_original=True)

# **Pipeline + Model**
______

In [17]:
os.chdir('/Users/dannyhogan/Desktop/ST-498/Code')


In [19]:
import pandas as pd
from ML_Pipeline import MLForecastingPipeline


pipeline = MLForecastingPipeline(target_col="US_Credit",n_splits=3, tune_splits=2)
pipeline.load_data(df)
pipeline.run()
pipeline.summary()
pipeline.best_params()

Data loaded: 107 rows, 9 columns.
Features: ['US_house_prices_lag1', 'US_cpi_lag1', 'US_unemployment_lag1', 'US_ExchangeRate_lag1', 'US_OilPrice_lag1', 'US_BondYield_lag1', 'US_rGDP_lag1', 'US_SP500_Open_lag1']

Running 7 models | 3 outer folds | 2 inner tuning folds



/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.245e-05, tolerance: 3.397e-05
  model = cd_fast.enet_coordinate_descent(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.768e-04, tolerance: 9.570e-05
  model = cd_fast.enet_coordinate_descent(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regula

  Fold 1 complete.


/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with f

  Fold 2 complete.


/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with f

  Fold 3 complete.

Done.

CV SUMMARY  (3-fold, averaged, tuned per fold)
           model    MAE   RMSE    MAPE   SMAPE       R2
    RandomForest 0.8725 1.0511 31.3423 26.5749 -21.4330
         XGBoost 1.0071 1.2301 38.1946 29.6062 -37.7473
        LightGBM 1.1452 1.2430 43.1134 33.5461 -49.0469
           Lasso 1.5409 1.7923 54.3647 68.4306 -18.3442
      ElasticNet 1.6474 2.0175 59.7087 70.1348 -19.2660
           Ridge 1.6972 2.1571 60.4610 72.3878 -61.8215
LinearRegression 1.8853 2.1883 70.6628 97.0365 -90.7400

Best model by RMSE : RandomForest
Best model by R²   : Lasso
 fold            model  alpha  l1_ratio  max_depth  n_estimators  learning_rate  subsample  num_leaves
    1       ElasticNet  0.010       0.5        NaN           NaN            NaN        NaN         NaN
    2       ElasticNet 10.000       0.8        NaN           NaN            NaN        NaN         NaN
    3       ElasticNet 10.000       0.2        NaN           NaN            NaN        NaN         NaN
    

/Users/dannyhogan/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,fold,model,alpha,l1_ratio,max_depth,n_estimators,learning_rate,subsample,num_leaves
3,1,ElasticNet,0.010,0.5,NaN,NaN,NaN,NaN,NaN
10,2,ElasticNet,10.000,0.8,NaN,NaN,NaN,NaN,NaN
17,3,ElasticNet,10.000,0.2,NaN,NaN,NaN,NaN,NaN
2,1,Lasso,0.001,NaN,NaN,NaN,NaN,NaN,NaN
9,2,Lasso,10.000,NaN,NaN,NaN,NaN,NaN,NaN
16,3,Lasso,10.000,NaN,NaN,NaN,NaN,NaN,NaN
6,1,LightGBM,NaN,NaN,NaN,50.0,0.01,0.8,15.0
13,2,LightGBM,NaN,NaN,NaN,50.0,0.01,0.8,15.0
20,3,LightGBM,NaN,NaN,NaN,200.0,0.10,0.8,15.0
0,1,LinearRegression,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
import os
print(os.getcwd())

/Users/dannyhogan/Desktop/ST-498


# **Modeling**
____

In [51]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from sklearn.model_selection import train_test_split, ParameterGrid, cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# metrics
from sklearn.metrics import * # accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve as sk_roc_curve
from scipy.stats import entropy

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet


### **Split the Data**

In [47]:
X = df.drop(columns=['UsRate'])
Y = df['UsRate']

tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(X, Y))

train_idx, test_idx = splits[0]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]

## **Linear Regression**
_____

In [ ]:
def linearModeling(pipeline, parameterGrid, x_train,x_test, y_train,y_test):
    if pipeline['model'] == 'LinearRegression':
        fitted = pipeline.fit(X_train,y_train)
        pred = fitted.predict(x_test)
        
    else:
        tscv = TimeSeriesSplit(n_splits=5)
        search = GridSearchCV(pipeline, parameterGrid,cv = tscv, n_jobs=-1)
        search.fit(x_train,y_train)
        pred = search.predict(x_test)
    mse = mean_squared_error(y_test,pred)
    print(mse)  



#### **OLS**

In [56]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
lr_params = {}  
lr_model = linearModeling(lr_pipeline, lr_params, X_train, X_test, y_train, y_test)


5.23282761109069


#### **Ridge**

In [57]:
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])
ridge_params = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}
ridge_model = linearModeling(ridge_pipeline, ridge_params, X_train, X_test, y_train, y_test)

0.9504402929631287


#### **Lasso**

In [58]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=10000))
])
lasso_params = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
}
lasso_model = linearModeling(lasso_pipeline, lasso_params, X_train, X_test, y_train, y_test)

0.24538468598029034


#### **Elastic Net**

In [59]:
elastic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(max_iter=10000))
])
elastic_params = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
elastic_model = linearModeling(elastic_pipeline, elastic_params, X_train, X_test, y_train, y_test)


0.26663093565756135


## **Tree-Based Methods**
___

In [ ]:
import sklearn.model_selection as ms
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

#### **Decision Tree Regression**

In [ ]:
def tree_modeling(name, pipeline, params, x_train, x_test, y_train, y_test):

    #Parameter tuning
    grid_search = GridSearchCV(pipeline, params, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(x_train, y_train)
    
    #Fitting best model
    best_model = grid_search.best_estimator_
    pred = best_model.predict(x_test)

    #Evaluation metrics
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    
    print(f"\n{name}")
    print(f"  Best Params: {grid_search.best_params_}")
    print(f"  MSE:  {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

    return best_model


#### **Decision Tree Regression**

In [ ]:
decision_tree = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeRegressor())
])
tree_params = {
    'model__max_depth': [None, 5, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}
tree_model = tree_modeling(decision_tree, X_train, X_test, y_train, y_test)

In [ ]:
random_forest = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor())
])
rf_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}
rf_model = tree_modeling(random_forest, X_train, X_test, y_train, y_test)   